# 练习实验：应用 Machine Learning 的建议

本实验将探索评估和改进 Machine Learning 模型的方法。

## 目录

- [1 - 软件包](#1)
- [2 - 评估学习算法：Polynomial Regression](#2)
  - [2.1 划分数据集](#2.1)
  - [2.2 计算 Linear Regression 的评估误差](#2.2)
    - [练习 1](#ex01)
  - [2.3 比较训练数据与测试数据上的表现](#2.3)
- [3 - Bias 与 Variance](#3)
  - [3.1 绘制 Training、Cross-validation 与 Test 数据](#3.1)
  - [3.2 寻找最佳 Polynomial Degree](#3.2)
  - [3.3 调整 Regularization](#3.3)
  - [3.4 获取更多数据：增大 Training Set Size](#3.4)
- [4 - 评估学习算法：Neural Network](#4)
  - [4.1 数据集](#4.1)
  - [4.2 使用 Classification Error 评估分类模型](#4.2)
    - [练习 2](#ex02)
- [5 - Model Complexity](#5)
  - [5.1 复杂模型](#complex-model)
    - [练习 3](#ex03)
  - [5.2 简单模型](#5.1)
    - [练习 4](#ex04)
- [6 - Regularization](#6)
  - [练习 5](#ex05)
- [7 - 迭代寻找最佳 Regularization Parameter](#7)
  - [7.1 测试](#7.1)

<img src="./images/C2_W3_BiasVarianceDegree.png" style="width:500px; padding:10px 20px;" alt="Polynomial Degree 与 Bias、Variance 的关系">


> **注意：** 为避免 autograder（自动评分器）报错，提交作业时请勿编辑或删除本 Notebook 中不参与评分的单元，也不要添加新单元。
>
> **通过本作业后**，如果想尝试修改不参与评分的代码，可以按照 Notebook 底部的说明操作。


<a name="1"></a>
## 1 - 软件包

运行下面的单元，导入本作业需要的软件包。

- [NumPy](https://numpy.org/)：Python 科学计算的基础库。
- [Matplotlib](http://matplotlib.org)：常用的 Python 绘图库。
- [Scikit-learn](https://scikit-learn.org/stable/)：用于数据挖掘与 Machine Learning 的库。
- [TensorFlow](https://www.tensorflow.org/)：常用的 Machine Learning 平台。


In [ ]:
import numpy as np
%matplotlib widget
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.activations import relu,linear
from tensorflow.keras.losses import SparseCategoricalCrossentropy
from tensorflow.keras.optimizers import Adam

import logging
logging.getLogger("tensorflow").setLevel(logging.ERROR)

from public_tests_a1 import * 

tf.keras.backend.set_floatx('float64')
from assigment_utils import *

tf.autograph.set_verbosity(0)

<a name="2"></a>
## 2 - 评估学习算法：Polynomial Regression

<img align="right" src="./images/C2_W3_TrainingVsNew.png" style="width:350px; padding:10px 20px;">

假设你创建了一个 Machine Learning 模型，它对训练数据拟合得很好。这还不意味着任务完成，因为模型的目标是预测**新样本**。

部署前如何测试模型对新数据的表现？需要两步：

- 将原始数据划分为 Training Set 和 Test Set，前者用于拟合模型参数，后者用于评估模型对未见过的数据的表现。
- 定义一个 Error Function，用于量化模型表现。


<a name="2.1"></a>
### 2.1 划分数据集

课程建议留出 20%–40% 的数据用于测试。下面使用 `sklearn` 的 [`train_test_split`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) 进行划分。运行后检查各数组的形状。


In [ ]:
# Generate some data
X,y,x_ideal,y_ideal = gen_data(18, 2, 0.7)
print("X.shape", X.shape, "y.shape", y.shape)

#split the data using sklearn routine 
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.33, random_state=1)
print("X_train.shape", X_train.shape, "y_train.shape", y_train.shape)
print("X_test.shape", X_test.shape, "y_test.shape", y_test.shape)

#### 2.1.1 绘制 Training Set 与 Test Set

图中红色训练样本与未参与训练的测试样本交错分布。该数据集由一个二次函数加上噪声生成，同时展示了“理想”曲线作为参考。


In [ ]:
fig, ax = plt.subplots(1,1,figsize=(4,4))
ax.plot(x_ideal, y_ideal, "--", color = "orangered", label="y_ideal", lw=1)
ax.set_title("Training, Test",fontsize = 14)
ax.set_xlabel("x")
ax.set_ylabel("y")

ax.scatter(X_train, y_train, color = "red",           label="train")
ax.scatter(X_test, y_test,   color = dlc["dlblue"],   label="test")
ax.legend(loc='upper left')
plt.show()

<a name="2.2"></a>
### 2.2 计算 Linear Regression 的评估误差

评估 Linear Regression 模型时，计算预测值与目标值之间的平方误差，并按本课程的约定除以 $2m$：

$$J_{\text{test}}(\mathbf{w},b)=\frac{1}{2m_{\text{test}}}\sum_{i=0}^{m_{\text{test}}-1}\left(f_{\mathbf{w},b}(\mathbf{x}_{\text{test}}^{(i)})-y_{\text{test}}^{(i)}\right)^2.\tag{1}$$


<a name="ex01"></a>
### 练习 1

完成下面的函数，计算 Linear Regression 模型在给定数据集上的误差。


In [ ]:
# UNQ_C1
# GRADED CELL: eval_mse
def eval_mse(y, yhat):
    """ 
    Calculate the mean squared error on a data set.
    Args:
      y    : (ndarray  Shape (m,) or (m,1))  target value of each example
      yhat : (ndarray  Shape (m,) or (m,1))  predicted value of each example
    Returns:
      err: (scalar)             
    """
    m = len(y)
    err = 0.0
    for i in range(m):
    ### START CODE HERE ### 
       
        
    
    ### END CODE HERE ### 
    
    return(err)

In [ ]:
y_hat = np.array([2.4, 4.2])
y_tmp = np.array([2.3, 4.1])
eval_mse(y_hat, y_tmp)

# BEGIN UNIT TEST
test_eval_mse(eval_mse)   
# END UNIT TEST

<details>
<summary><strong>点击查看提示</strong></summary>

```python
def eval_mse(y, yhat):
    """ 
    Calculate the mean squared error on a data set.
    Args:
      y    : (ndarray  Shape (m,) or (m,1))  target value of each example
      yhat : (ndarray  Shape (m,) or (m,1))  predicted value of each example
    Returns:
      err: (scalar)             
    """
    m = len(y)
    err = 0.0
    for i in range(m):
        err_i  = ( (yhat[i] - y[i])**2 ) 
        err   += err_i                                                                
    err = err / (2*m)                    
    return(err)
```

</details>


<a name="2.3"></a>
### 2.3 比较训练数据与测试数据上的表现

构建一个高次 Polynomial Regression 模型，以尽量降低训练误差。这里使用 `sklearn` 的 Linear Regression 功能；具体实现可在导入的辅助文件中查看。步骤如下：

- 创建并拟合模型；“拟合”也就是训练模型，Gradient Descent 是一种训练方法。
- 计算 Training Set 上的误差。
- 计算 Test Set 上的误差。


In [ ]:
# create a model in sklearn, train on training data
degree = 10
lmodel = lin_model(degree)
lmodel.fit(X_train, y_train)

# predict on training data, find training error
yhat = lmodel.predict(X_train)
err_train = lmodel.mse(y_train, yhat)

# predict on test data, find error
yhat = lmodel.predict(X_test)
err_test = lmodel.mse(y_test, yhat)

Training Set 上的误差明显小于 Test Set 上的误差。


In [ ]:
print(f"training err {err_train:0.2f}, test err {err_test:0.2f}")

下面的图解释了原因：模型构造出复杂的函数，以很好地拟合训练数据，但对未参与训练的测试数据预测较差。这种模型存在 Overfitting，具有 High Variance，Generalization 能力较弱。


In [ ]:
# plot predictions over data range 
x = np.linspace(0,int(X.max()),100)  # predict values for plot
y_pred = lmodel.predict(x).reshape(-1,1)

plt_train_test(X_train, y_train, X_test, y_test, x, y_pred, x_ideal, y_ideal, degree)

Test Error 表明，这个模型在新数据上表现不佳。如果使用 Test Error 指导改进，模型可能逐渐适应该 Test Set；它便不能再独立代表“新数据”。因此，还需要一组独立数据评估最终表现。

课程建议将数据分为三部分。下表给出一种常见比例，可根据可用数据量调整。

| 数据集 | 占比 | 用途 |
| --- | :---: | --- |
| Training Set | 60% | 训练时拟合参数 $w$ 和 $b$ |
| Cross-validation Set | 20% | 调整 Polynomial Degree、Regularization、Neural Network 架构等 Hyperparameters |
| Test Set | 20% | 调参结束后，评估模型在新数据上的表现 |

下面调用两次 `sklearn` 的 `train_test_split`，得到这三个数据集。


In [ ]:
# Generate  data
X,y, x_ideal,y_ideal = gen_data(40, 5, 0.7)
print("X.shape", X.shape, "y.shape", y.shape)

#split the data using sklearn routine 
X_train, X_, y_train, y_ = train_test_split(X,y,test_size=0.40, random_state=1)
X_cv, X_test, y_cv, y_test = train_test_split(X_,y_,test_size=0.50, random_state=1)
print("X_train.shape", X_train.shape, "y_train.shape", y_train.shape)
print("X_cv.shape", X_cv.shape, "y_cv.shape", y_cv.shape)
print("X_test.shape", X_test.shape, "y_test.shape", y_test.shape)

<a name="3"></a>
## 3 - Bias 与 Variance

<img align="right" src="./images/C2_W3_BiasVarianceDegree.png" style="width:500px; padding:10px 20px;">

上面的 Polynomial Degree 明显过高。如何选择合适的值？如图所示，可以比较训练表现与 Cross-validation 表现。尝试一系列次数并计算两类误差；当次数过高时，Cross-validation 表现会相对于训练表现开始恶化。下面在当前示例中验证。


<a name="3.1"></a>
### 3.1 绘制 Training、Cross-validation 与 Test 数据

下面可以看到，红色训练样本与未参与训练的 Cross-validation、Test 样本交错分布。


In [ ]:
fig, ax = plt.subplots(1,1,figsize=(4,4))
ax.plot(x_ideal, y_ideal, "--", color = "orangered", label="y_ideal", lw=1)
ax.set_title("Training, CV, Test",fontsize = 14)
ax.set_xlabel("x")
ax.set_ylabel("y")

ax.scatter(X_train, y_train, color = "red",           label="train")
ax.scatter(X_cv, y_cv,       color = dlc["dlorange"], label="cv")
ax.scatter(X_test, y_test,   color = dlc["dlblue"],   label="test")
ax.legend(loc='upper left')
plt.show()

<a name="3.2"></a>
### 3.2 寻找最佳 Polynomial Degree

此前的实验表明，Polynomial Features 能让模型拟合复杂曲线（参见 Course 1 Week 2 的 Feature Engineering 与 Polynomial Regression 实验），而提高次数可能造成 Overfitting（参见 Course 1 Week 3 的 Overfitting 实验）。现在利用这些知识，练习区分 Overfitting 和 Underfitting。

反复训练模型，每次增加 Polynomial Degree。为简化实现并提高速度，这里使用 [Scikit-learn 的 LinearRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html#sklearn.linear_model.LinearRegression)。


In [ ]:
max_degree = 9
err_train = np.zeros(max_degree)    
err_cv = np.zeros(max_degree)      
x = np.linspace(0,int(X.max()),100)  
y_pred = np.zeros((100,max_degree))  #columns are lines to plot

for degree in range(max_degree):
    lmodel = lin_model(degree+1)
    lmodel.fit(X_train, y_train)
    yhat = lmodel.predict(X_train)
    err_train[degree] = lmodel.mse(y_train, yhat)
    yhat = lmodel.predict(X_cv)
    err_cv[degree] = lmodel.mse(y_cv, yhat)
    y_pred[:,degree] = lmodel.predict(x)
    
optimal_degree = np.argmin(err_cv)+1

下面绘制结果：


In [ ]:
plt.close("all")
plt_optimal_degree(X_train, y_train, X_cv, y_cv, x, y_pred, x_ideal, y_ideal, 
                   err_train, err_cv, optimal_degree, max_degree)

上图表明，将数据分为参与训练和未参与训练的两组，可以帮助判断 Underfitting 或 Overfitting。本例通过逐步增加 Polynomial Degree，构造了从欠拟合到过拟合的多个模型。

- 左图中的实线表示各模型的预测。一次多项式是一条直线，只经过少量数据点；最高次模型则紧贴几乎所有训练点。
- 右图中，蓝色训练误差随模型复杂度增大而下降，符合预期。Cross-validation Error 起初随拟合改善而下降，随后因 Overfitting、Generalization 变差而上升。

实际曲线没有课程示意图那么平滑。分配到各组的具体数据点会显著影响结果，应重点观察整体趋势。


<a name="3.3"></a>
### 3.3 调整 Regularization

此前的实验使用 Regularization 减轻 Overfitting。与选择 Polynomial Degree 类似，可以通过 Training Set 和 Cross-validation Set 调整 Regularization Parameter $\lambda$。

下面固定一个较高的 Polynomial Degree，改变 $\lambda$ 进行比较。


In [ ]:
lambda_range = np.array([0.0, 1e-6, 1e-5, 1e-4,1e-3,1e-2, 1e-1,1,10,100])
num_steps = len(lambda_range)
degree = 10
err_train = np.zeros(num_steps)    
err_cv = np.zeros(num_steps)       
x = np.linspace(0,int(X.max()),100) 
y_pred = np.zeros((100,num_steps))  #columns are lines to plot

for i in range(num_steps):
    lambda_= lambda_range[i]
    lmodel = lin_model(degree, regularization=True, lambda_=lambda_)
    lmodel.fit(X_train, y_train)
    yhat = lmodel.predict(X_train)
    err_train[i] = lmodel.mse(y_train, yhat)
    yhat = lmodel.predict(X_cv)
    err_cv[i] = lmodel.mse(y_cv, yhat)
    y_pred[:,i] = lmodel.predict(x)
    
optimal_reg_idx = np.argmin(err_cv) 

In [ ]:
plt.close("all")
plt_tune_regularization(X_train, y_train, X_cv, y_cv, x, y_pred, err_train, err_cv, optimal_reg_idx, lambda_range)

上图显示，随着 Regularization 增强，模型从 High Variance（Overfitting）逐渐转为 High Bias（Underfitting）。右图的竖线标出最佳 $\lambda$。本例的 Polynomial Degree 固定为 10。


<a name="3.4"></a>
### 3.4 获取更多数据：增大 Training Set Size（$m$）

当模型存在 Overfitting（High Variance）时，增加训练数据可能改善表现。下面尝试这一方法。


In [ ]:
X_train, y_train, X_cv, y_cv, x, y_pred, err_train, err_cv, m_range,degree = tune_m()
plt_tune_m(X_train, y_train, X_cv, y_cv, x, y_pred, err_train, err_cv, m_range, degree)

上图显示，对于 High Variance、Overfitting 的模型，增加样本能改善表现。左图中，最大 $m$ 对应的最终曲线较为平滑，并位于数据的中间。右图中，随着样本数增加，训练误差与 Cross-validation Error 趋于接近。

实际曲线不如课堂示意图平滑是正常的，整体趋势仍很清楚：更多数据改善了 Generalization。

> **注意：** 对于 High Bias（Underfitting），单纯增加样本通常无法改善表现。


<a name="4"></a>
## 4 - 评估学习算法：Neural Network

前面调整了 Polynomial Regression 模型，接下来处理 Neural Network。首先创建一个 Classification 数据集。


<a name="4.1"></a>
### 4.1 数据集

运行下面的单元生成数据，并划分为 Training Set、Cross-validation Set（CV）与 Test Set。为突出比较效果，本例提高了 CV 数据的比例。


In [ ]:
# Generate and split data set
X, y, centers, classes, std = gen_blobs()

# split the data. Large CV population for demonstration
X_train, X_, y_train, y_ = train_test_split(X,y,test_size=0.50, random_state=1)
X_cv, X_test, y_cv, y_test = train_test_split(X_,y_,test_size=0.20, random_state=1)
print("X_train.shape:", X_train.shape, "X_cv.shape:", X_cv.shape, "X_test.shape:", X_test.shape)

In [ ]:
plt_train_eq_dist(X_train, y_train,classes, X_cv, y_cv, centers, std)

左图用颜色标出六个 Cluster，同时显示训练样本（圆点）与 CV 样本（三角形）。留意那些处于模糊区域、可能属于两个相邻 Cluster 的点。你期望 Neural Network 如何处理它们？怎样的结果代表 Overfitting 或 Underfitting？

右图展示了已知数据生成方式时可构造的“理想”模型。边界上的点到相邻 Cluster 中心的距离相等。即使这个模型，也会将全部数据中约 8% 的样本“分类错误”。


<a name="4.2"></a>
### 4.2 使用 Classification Error 评估分类模型

这里使用预测错误的样本占比作为评估指标：

$$J_{cv}=\frac{1}{m}\sum_{i=0}^{m-1}
\begin{cases}
1,&\hat{y}^{(i)}\ne y^{(i)},\\
0,&\text{otherwise}.
\end{cases}
$$

<a name="ex02"></a>
### 练习 2

完成下面的函数，计算 Classification Error。本实验的目标标签是类别索引，未使用 [One-hot Encoding](https://en.wikipedia.org/wiki/One-hot)。


In [ ]:
# UNQ_C2
# GRADED CELL: eval_cat_err
def eval_cat_err(y, yhat):
    """ 
    Calculate the categorization error
    Args:
      y    : (ndarray  Shape (m,) or (m,1))  target value of each example
      yhat : (ndarray  Shape (m,) or (m,1))  predicted value of each example
    Returns:|
      cerr: (scalar)             
    """
    m = len(y)
    incorrect = 0
    for i in range(m):
    ### START CODE HERE ### 
       
            
    
    ### END CODE HERE ### 
    
    return(cerr)

In [ ]:
y_hat = np.array([1, 2, 0])
y_tmp = np.array([1, 2, 3])
print(f"categorization error {np.squeeze(eval_cat_err(y_hat, y_tmp)):0.3f}, expected:0.333" )
y_hat = np.array([[1], [2], [0], [3]])
y_tmp = np.array([[1], [2], [1], [3]])
print(f"categorization error {np.squeeze(eval_cat_err(y_hat, y_tmp)):0.3f}, expected:0.250" )

# BEGIN UNIT TEST  
test_eval_cat_err(eval_cat_err)
# END UNIT TEST

<details>
<summary><strong>点击查看提示</strong></summary>

```python
def eval_cat_err(y, yhat):
    """ 
    Calculate the categorization error
    Args:
      y    : (ndarray  Shape (m,) or (m,1))  target value of each example
      yhat : (ndarray  Shape (m,) or (m,1))  predicted value of each example
    Returns:|
      cerr: (scalar)             
    """
    m = len(y)
    incorrect = 0
    for i in range(m):
        if yhat[i] != y[i]:    # @REPLACE
            incorrect += 1     # @REPLACE
    cerr = incorrect/m         # @REPLACE
    return(cerr)
```

</details>


<a name="5"></a>
## 5 - Model Complexity

下面分别构建复杂模型与简单模型，并通过评估判断它们是否可能 Overfit 或 Underfit。

<a name="complex-model"></a>
### 5.1 复杂模型

<a name="ex03"></a>
### 练习 3

构建一个三层模型：

- Dense Layer：120 个单元，ReLU Activation。
- Dense Layer：40 个单元，ReLU Activation。
- Dense Layer：6 个单元，Linear Activation。

调用 `compile` 时，使用 `SparseCategoricalCrossentropy(from_logits=True)` 作为 Loss Function，并使用 Learning Rate 为 `0.01` 的 Adam Optimizer。


In [ ]:
# UNQ_C3
# GRADED CELL: model
import logging
logging.getLogger("tensorflow").setLevel(logging.ERROR)

tf.random.set_seed(1234)
model = Sequential(
    [
        ### START CODE HERE ### 
        
        
        
        ### END CODE HERE ### 

    ], name="Complex"
)
model.compile(
    ### START CODE HERE ### 
    loss=None,
    optimizer=None,
    ### END CODE HERE ### 
)

In [ ]:
# BEGIN UNIT TEST
model.fit(
    X_train, y_train,
    epochs=1000
)
# END UNIT TEST

In [ ]:
# BEGIN UNIT TEST
model.summary()

model_test(model, classes, X_train.shape[1]) 
# END UNIT TEST

<details>
<summary><strong>点击查看提示</strong></summary>

模型摘要应与下方一致，层实例名称的编号可能不同。

```
Model: "Complex"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
=================================================================
L1 (Dense)                   (None, 120)               360       
_________________________________________________________________
L2 (Dense)                   (None, 40)                4840      
_________________________________________________________________
L3 (Dense)                   (None, 6)                 246       
=================================================================
Total params: 5,446
Trainable params: 5,446
Non-trainable params: 0
_________________________________________________________________
```

<details>
<summary><strong>点击查看更多提示</strong></summary>

```python
tf.random.set_seed(1234)
model = Sequential(
    [
        Dense(120, activation = 'relu', name = "L1"),      
        Dense(40, activation = 'relu', name = "L2"),         
        Dense(classes, activation = 'linear', name = "L3")  
    ], name="Complex"
)
model.compile(
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),          
    optimizer=tf.keras.optimizers.Adam(0.01),   
)

model.fit(
    X_train,y_train,
    epochs=1000
)
```

</details>
</details>


In [ ]:
#make a model for plotting routines to call
model_predict = lambda Xl: np.argmax(tf.nn.softmax(model.predict(Xl)).numpy(),axis=1)
plt_nn(model_predict,X_train,y_train, classes, X_cv, y_cv, suptitle="Complex Model")

这个模型努力拟合各类别中的 Outliers，结果将部分 CV 样本错误分类。下面计算 Classification Error。


In [ ]:
training_cerr_complex = eval_cat_err(y_train, model_predict(X_train))
cv_cerr_complex = eval_cat_err(y_cv, model_predict(X_cv))
print(f"categorization error, training, complex model: {training_cerr_complex:0.3f}")
print(f"categorization error, cv,       complex model: {cv_cerr_complex:0.3f}")

<a name="5.1"></a>
### 5.2 简单模型

接下来尝试一个简单模型。

<a name="ex04"></a>
### 练习 4

构建一个两层模型：

- Dense Layer：6 个单元，ReLU Activation。
- Dense Layer：6 个单元，Linear Activation。

调用 `compile` 时，使用 `SparseCategoricalCrossentropy(from_logits=True)` 作为 Loss Function，并使用 Learning Rate 为 `0.01` 的 Adam Optimizer。


In [ ]:
# UNQ_C4
# GRADED CELL: model_s

tf.random.set_seed(1234)
model_s = Sequential(
    [
        ### START CODE HERE ### 
        
        
        ### END CODE HERE ### 
    ], name = "Simple"
)
model_s.compile(
    ### START CODE HERE ### 
    loss=None,
    optimizer=None,
    ### START CODE HERE ### 
)


In [ ]:
import logging
logging.getLogger("tensorflow").setLevel(logging.ERROR)

# BEGIN UNIT TEST
model_s.fit(
    X_train,y_train,
    epochs=1000
)
# END UNIT TEST

In [ ]:
# BEGIN UNIT TEST
model_s.summary()

model_s_test(model_s, classes, X_train.shape[1])
# END UNIT TEST

<details>
<summary><strong>点击查看提示</strong></summary>

模型摘要应与下方一致，层实例名称的编号可能不同。

```
Model: "Simple"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
=================================================================
L1 (Dense)                   (None, 6)                 18        
_________________________________________________________________
L2 (Dense)                   (None, 6)                 42        
=================================================================
Total params: 60
Trainable params: 60
Non-trainable params: 0
_________________________________________________________________
```

<details>
<summary><strong>点击查看更多提示</strong></summary>

```python
tf.random.set_seed(1234)
model_s = Sequential(
    [
        Dense(6, activation = 'relu', name="L1"),            # @REPLACE
        Dense(classes, activation = 'linear', name="L2")     # @REPLACE
    ], name = "Simple"
)
model_s.compile(
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),     # @REPLACE
    optimizer=tf.keras.optimizers.Adam(0.01),     # @REPLACE
)

model_s.fit(
    X_train,y_train,
    epochs=1000
)
```

</details>
</details>


In [ ]:
#make a model for plotting routines to call
model_predict_s = lambda Xl: np.argmax(tf.nn.softmax(model_s.predict(Xl)).numpy(),axis=1)
plt_nn(model_predict_s,X_train,y_train, classes, X_cv, y_cv, suptitle="Simple Model")

这个简单模型表现不错。下面计算 Classification Error。


In [ ]:
training_cerr_simple = eval_cat_err(y_train, model_predict_s(X_train))
cv_cerr_simple = eval_cat_err(y_cv, model_predict_s(X_cv))
print(f"categorization error, training, simple model, {training_cerr_simple:0.3f}, complex model: {training_cerr_complex:0.3f}" )
print(f"categorization error, cv,       simple model, {cv_cerr_simple:0.3f}, complex model: {cv_cerr_complex:0.3f}" )

与复杂模型相比，简单模型的训练 Classification Error 略高，但在 Cross-validation Set 上表现更好。


<a name="6"></a>
## 6 - Regularization

与 Polynomial Regression 类似，也可以用 Regularization 限制复杂模型的拟合程度。下面进行尝试。

<a name="ex05"></a>
### 练习 5

重新构建复杂模型，并加入 Regularization：

- Dense Layer：120 个单元，ReLU Activation，设置 `kernel_regularizer=tf.keras.regularizers.l2(0.1)`。
- Dense Layer：40 个单元，ReLU Activation，设置 `kernel_regularizer=tf.keras.regularizers.l2(0.1)`。
- Dense Layer：6 个单元，Linear Activation。

调用 `compile` 时，使用 `SparseCategoricalCrossentropy(from_logits=True)` 作为 Loss Function，并使用 Learning Rate 为 `0.01` 的 Adam Optimizer。


In [ ]:
# UNQ_C5
# GRADED CELL: model_r

tf.random.set_seed(1234)
model_r = Sequential(
    [
        ### START CODE HERE ### 
        
        
        
        ### START CODE HERE ### 
    ], name= None
)
model_r.compile(
    ### START CODE HERE ### 
    loss=None,
    optimizer=None,
    ### START CODE HERE ### 
)


In [ ]:
# BEGIN UNIT TEST
model_r.fit(
    X_train, y_train,
    epochs=1000
)
# END UNIT TEST

In [ ]:
# BEGIN UNIT TEST
model_r.summary()

model_r_test(model_r, classes, X_train.shape[1]) 
# END UNIT TEST

<details>
<summary><strong>点击查看提示</strong></summary>

模型摘要应与下方一致，层实例名称的编号可能不同。

```
Model: "ComplexRegularized"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
=================================================================
L1 (Dense)                   (None, 120)               360       
_________________________________________________________________
L2 (Dense)                   (None, 40)                4840      
_________________________________________________________________
L3 (Dense)                   (None, 6)                 246       
=================================================================
Total params: 5,446
Trainable params: 5,446
Non-trainable params: 0
_________________________________________________________________
```

<details>
<summary><strong>点击查看更多提示</strong></summary>

```python
tf.random.set_seed(1234)
model_r = Sequential(
    [
        Dense(120, activation = 'relu', kernel_regularizer=tf.keras.regularizers.l2(0.1), name="L1"), 
        Dense(40, activation = 'relu', kernel_regularizer=tf.keras.regularizers.l2(0.1), name="L2"),  
        Dense(classes, activation = 'linear', name="L3")  
    ], name="ComplexRegularized"
)
model_r.compile(
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True), 
    optimizer=tf.keras.optimizers.Adam(0.01),                             
)

model_r.fit(
    X_train,y_train,
    epochs=1000
)
```

</details>
</details>


In [ ]:
#make a model for plotting routines to call
model_predict_r = lambda Xl: np.argmax(tf.nn.softmax(model_r.predict(Xl)).numpy(),axis=1)
 
plt_nn(model_predict_r, X_train,y_train, classes, X_cv, y_cv, suptitle="Regularized")

结果与“理想”模型十分接近。下面检查 Classification Error。


In [ ]:
training_cerr_reg = eval_cat_err(y_train, model_predict_r(X_train))
cv_cerr_reg = eval_cat_err(y_cv, model_predict_r(X_cv))
test_cerr_reg = eval_cat_err(y_test, model_predict_r(X_test))
print(f"categorization error, training, regularized: {training_cerr_reg:0.3f}, simple model, {training_cerr_simple:0.3f}, complex model: {training_cerr_complex:0.3f}" )
print(f"categorization error, cv,       regularized: {cv_cerr_reg:0.3f}, simple model, {cv_cerr_simple:0.3f}, complex model: {cv_cerr_complex:0.3f}" )

简单模型在 Training Set 上略优于加入 Regularization 的模型，但在 Cross-validation Set 上表现较差。


<a name="7"></a>
## 7 - 迭代寻找最佳 Regularization Parameter

与 Linear Regression 一样，可以尝试多个 Regularization Parameter。下面的代码需要运行几分钟。时间允许时可以运行并查看结果；到这里，作业的评分部分已经完成。


In [ ]:
tf.random.set_seed(1234)
lambdas = [0.0, 0.001, 0.01, 0.05, 0.1, 0.2, 0.3]
models=[None] * len(lambdas)

for i in range(len(lambdas)):
    lambda_ = lambdas[i]
    models[i] =  Sequential(
        [
            Dense(120, activation = 'relu', kernel_regularizer=tf.keras.regularizers.l2(lambda_)),
            Dense(40, activation = 'relu', kernel_regularizer=tf.keras.regularizers.l2(lambda_)),
            Dense(classes, activation = 'linear')
        ]
    )
    models[i].compile(
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        optimizer=tf.keras.optimizers.Adam(0.01),
    )

    models[i].fit(
        X_train,y_train,
        epochs=1000
    )
    print(f"Finished lambda = {lambda_}")


In [ ]:
plot_iterate(lambdas, models, X_train, y_train, X_cv, y_cv)

随着 Regularization 增强，训练表现与 Cross-validation 表现逐渐接近。对于本数据集和模型，$\lambda>0.01$ 看起来是合理的选择。


<a name="7.1"></a>
### 7.1 测试

在 Test Set 上评估优化后的模型，并与“理想”模型比较。


In [ ]:
plt_compare(X_test,y_test, classes, model_predict_s, model_predict_r, centers)

Test Set 较小，且似乎包含一些 Outliers，因此 Classification Error 较高。不过，优化后的模型表现与“理想”模型相当。


## 恭喜完成！

你已经熟悉了评估 Machine Learning 模型的重要方法：

- 将数据分为参与训练和未参与训练的集合，有助于区分 Underfitting 与 Overfitting。
- 用 Training Set 拟合参数 $W$、$b$，用 Cross-validation Set 调整 Model Complexity、Regularization、训练样本数量等选择，最后用 Test Set 评估对新数据的表现。
- 比较训练表现与 Cross-validation 表现，可以判断模型更倾向于 High Variance（Overfitting）还是 High Bias（Underfitting）。


<details>
<summary><span style="color:darkgreen;"><strong>如需尝试修改不参与评分的代码，请点击这里。</strong></span></summary>

**重要提示：请在通过作业后再执行这些操作，以免影响 autograder。**

1. 在 Notebook 菜单中，依次选择 `View` > `Cell Toolbar` > `Edit Metadata`。
2. 在希望锁定或解锁的代码单元旁，点击 `Edit Metadata`。
3. 将 `editable` 属性设为 `true` 以解锁，或设为 `false` 以锁定。
4. 在 Notebook 菜单中，依次选择 `View` > `Cell Toolbar` > `None`。

下面的简短演示展示了上述步骤：

<img src="https://lh3.google.com/u/0/d/14Xy_Mb17CZVgzVAgq7NCjMVBvSae3xO1" alt="解锁 Notebook 单元的操作演示（unlock_cells.gif）">

</details>
